# 閾値（しきい値）トリガによるデータ取得 — DMA + Plotly 版

このノートブックは `detector-acquisition.ipynb` の **後半（Option B：全波形ロギング）** を **DMA (Deep Memory Acquisition / AXI)** で書き直し、プロットには numpy 依存が緩い **Plotly** を採用したものです。

- **Option A** (CSV ピーク・積分ロギング) → **そのまま** 標準 ACQ (`rp_AcqGetOldestDataV`) を使います。
- **Option B** (`.npz` 全波形ロギング) → **DMA** (`rp_AcqAxi*`) に置き換え。データを Linux カーネルバッファを経由せず DDR に直接書き込むので、長いバッファでも取りこぼしが起きにくくなります。

**注意:**
Red Pitaya の高速アナログ入力の電圧レンジは、本体上の **入力ジャンパ** の位置で決まります。

- **HV** 設定 → 入力レンジ ±20 V
- **LV** 設定 → 入力レンジ ±1 V

詳しくは公式ドキュメントの[こちらの章](https://redpitaya.readthedocs.io/en/latest/developerGuide/hardware/125-14/fastIO.html#analog-inputs)をご参照ください。

下の図のように、**高速アナログ出力 (OUT) と高速アナログ入力 (IN) を SMA ケーブルで直結（ループバック接続）** してください。ジャンパは **±1 V (LV)** に設定されていることを必ず確認してください。

![Fast loop back](../img/FastIOLoopBack.png "ループバック接続の例")

## DMA とは何か

Deep Memory Acquisition (DMA) は、Red Pitaya の **DDR メモリに直接書き込む** タイプのデータ取得です。標準の `rp_AcqGetOldestData*` 系 API との違いは:

- バッファ長が **16384 サンプル固定ではない** (任意のサンプル数。64 バイト境界に揃える必要あり)
- AXI 経由で DDR に直接ストリーミング (Linux 側の DMA 予約領域を共有)
- フルレート 125 MSPS で取得可能
- 取り出しに時間がかかる (バッファが大きいほど顕著)
- **標準 ACQ と並行して動作可能** ← 本ノートブックの Option A と Option B を切り替えて使えるのはこのおかげ

本ノートブックでは **1 ショット = 16384 サンプル/ch (オリジナルと同じ長さ)** で DMA を構成します。チャンネルあたり 32 KB、合計 64 KB なので、Linux 側の DDR 圧迫はほぼゼロです。詳細は[公式ドキュメント](https://redpitaya.readthedocs.io/en/latest/appsFeatures/remoteControl/deepMemoryAcquisition.html)を参照してください。

## 0. Plotly のインストール (最初に 1 回だけ実行)

Red Pitaya のシステム Python (3.10) で動作確認済みのバージョン組み合わせです。`numpy` を巻き込んで再ビルドが走らないように、すべて **バイナリ wheel が配布済み** のバージョンに固定し、`--no-deps` で純 Python 依存だけ別途入れます。

下のセルを **1 回だけ** 実行 → カーネルを再起動してください。インストール済みなら 2 回目以降はスキップしてかまいません。

```bash
# シェルで実行する場合 (Red Pitaya に SSH した状態で)
pip install --user --no-deps "plotly==5.24.1"
pip install --user "tenacity>=8.2.0,<9" "packaging>=24.0"
```

ポイント:
- `--no-deps` を付けると、Plotly のサブ依存が新しい numpy を要求して再ビルドが走るのを防げます。
- `--user` を付けると `~/.local/` 以下に入るため、システムの site-packages を汚しません。
- `tenacity`, `packaging` は Plotly 5.24 が動作するために必要な純 Python の依存だけ別途入れています。

In [ ]:
# ノートブックから直接インストールしたい場合はこのセル (1 回だけ)。既に入っている場合はスキップ。
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "--no-deps", "plotly==5.24.1"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "tenacity>=8.2.0,<9", "packaging>=24.0"])
print("Plotly インストール完了。カーネルを一度再起動してから次のセルへ進んでください。")

## ライブラリと FPGA イメージの読み込み

データのプロットに **Plotly**、配列演算に `numpy` を使います。Plotly は numpy への依存が緩いので、システムの numpy が古くても新しくても動きます。Red Pitaya 固有の `rp_overlay`（FPGA イメージ読み込み）と `rp`（API 本体）も読み込みます。

In [ ]:
import time
import datetime
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import importlib, paper_style
importlib.reload(paper_style)  # pick up edits to paper_style.py without kernel restart
from rp_overlay import overlay
import rp
fpga = overlay()
rp.rp_Init()
print("FPGA ready, plotly", __import__('plotly').__version__)

## マクロ一覧

標準 ACQ で使うマクロに加え、DMA では以下も登場します。

- **デシメーション (Decimation)** — `RP_DEC_1`, `RP_DEC_2`, `RP_DEC_4`, ... `RP_DEC_65536` (DMA は 1〜65536 の任意整数も可)
- **取得トリガ (Acquisition trigger)** — `RP_TRIG_SRC_DISABLED`, `RP_TRIG_SRC_NOW`, `RP_TRIG_SRC_CHA_PE`, `RP_TRIG_SRC_CHA_NE`, `RP_TRIG_SRC_CHB_PE`, `RP_TRIG_SRC_CHB_NE`, `RP_TRIG_SRC_EXT_PE`, `RP_TRIG_SRC_EXT_NE`, `RP_TRIG_SRC_AWG_PE`, `RP_TRIG_SRC_AWG_NE`
- **トリガ状態 (Acquisition trigger state)** — `RP_TRIG_STATE_TRIGGERED`, `RP_TRIG_STATE_WAITING`
- **バッファサイズ (Buffer size)** — `ADC_BUFFER_SIZE`, `DAC_BUFFER_SIZE`
- **高速アナログチャンネル (Fast analog channels)** — `RP_CH_1`, `RP_CH_2`

**DMA 専用 API (一部)**
- `rp_AcqAxiGetMemoryRegion()` → `[status, start_addr, size_samples]`
- `rp_AcqAxiSetDecimationFactor(dec)`
- `rp_AcqAxiSetTriggerDelay(ch, samples_after_trigger)`
- `rp_AcqAxiSetBufferSamples(ch, addr, samples)`
- `rp_AcqAxiEnable(ch, bool)`
- `rp_AcqAxiGetBufferFillState(ch)` / `rp_AcqAxiGetWritePointerAtTrig(ch)`
- `rp_AcqAxiGetDataRaw(ch, pos, size, buf)` ← データ読み出し (RAW, int16)

## 用語の説明

データ取得（アクイジション）を正しく設定するためには、いくつかの専門用語を知っておく必要があります。

- **トリガモーメント (Triggering moment)** — トリガ条件が満たされ、装置がデータの取り込みを開始する瞬間です。
- **トリガ条件 (Trigger condition)** — 「トリガレベル」と「シグナルフロント」の組み合わせで決まります。
- **トリガレベル (Trigger level)** — 入力信号がこの電圧 [V] に達したらデータ取得を始める、という閾値の値です。
- **シグナルフロント (Signal front)** — **立ち上がりエッジ／立ち下がりエッジ (positive/negative edge)** とも呼ばれます。
- **デシメーション (Decimation, 間引き)** — 何サンプルおきに 1 サンプル保存するかの設定です。
- **平均化 (Averaging)** — デシメーションが 2 以上のとき、保存される 1 サンプルは「捨てる側のサンプルすべての平均」になります。
- **取得単位 (Acquisition units)** — *RAW*（ADC の生値）と *Volts*（電圧 [V]）。**DMA は RAW のみ** で、本ノートブックでは ±1 V (LV) 換算を手動で行います (`raw / 8192`)。
- **トリガディレイ (Trigger delay)** — 標準 ACQ ではバッファ内のトリガ位置をシフトする値、**DMA では「トリガ後に何サンプル取り続けるか」** を意味します (意味が異なるので注意)。

## トリガレベル等の設定 (標準 ACQ)

取得パラメータ (デシメーション、トリガレベル、ジャンパ位置に応じた LSB 換算係数など) を 1 か所にまとめて宣言し、Option A や動作確認プロットで使う標準 ACQ 用の `acqADC()` を定義します。`acqADC()` と Option B の `acqADC_DMA()` は内部で共通の `_start_and_wait()` を呼びます。

In [ ]:
# === 設定 (Config) ============================================================
# 取得 (Acquisition) パラメータ
dec = rp.RP_DEC_1                # デシメーション (1 -> 125 MS/s, そのまま保存)
N = 16384                        # 取得バッファのサンプル数 (固定)
trigger_ch = 0                   # トリガに使うチャンネルのインデックス (0 -> CH1, 1 -> CH2)
trig_lvl = 0.10                  # トリガレベル [V]
trig_dly = int(N / 2) - 30       # 標準 ACQ 用トリガディレイ [samples]: トリガ瞬間をバッファ先頭から 30 サンプル目に置く
duty = int(N / 64)               # Option A で実際に使う波形長 [samples] (バッファ先頭から duty サンプルだけ抜き出す)

# DMA パラメータ
DMA_DATA_SIZE   = N                              # 1 ショットのサンプル数 (16384、64 バイト境界 OK)
DMA_PRE_TRIGGER = 256                            # 各ショットで「トリガ瞬間より前」に残すサンプル数
DMA_TRIG_DELAY  = DMA_DATA_SIZE - DMA_PRE_TRIGGER  # トリガ後に取り続けるサンプル数

# DMA パイプライン固定レイテンシ補正 (samples)。0 なら未較正。較正セルで上書きされる。
FIXED_OFFSET_SAMPLES = 0

# 入力ジャンパに応じた LSB → V 換算係数 (DMA は RAW しか返さないので手動で換算する)
INPUT_JUMPER = "LV"              # "LV" (±1V) または "HV" (±20V)
LSB_VOLTS_BY_JUMPER = {"LV": 1.0 / 8192.0, "HV": 20.0 / 8192.0}
_DMA_LSB_VOLTS = LSB_VOLTS_BY_JUMPER[INPUT_JUMPER]

# トリガ／チャンネルマッピング
trigger_list = [rp.RP_T_CH_1, rp.RP_T_CH_2]                       # トリガチャンネル定数
acq_trig_sour_list = [rp.RP_TRIG_SRC_CHA_PE, rp.RP_TRIG_SRC_CHB_PE]  # 取得用トリガソース (index 0 -> CH1)

# トリガソース定数 → 名称 (メタデータ JSON に正しい値を残すための逆引き表)
TRIG_SRC_NAMES = {
    rp.RP_TRIG_SRC_DISABLED: "RP_TRIG_SRC_DISABLED",
    rp.RP_TRIG_SRC_NOW:      "RP_TRIG_SRC_NOW",
    rp.RP_TRIG_SRC_CHA_PE:   "RP_TRIG_SRC_CHA_PE",
    rp.RP_TRIG_SRC_CHA_NE:   "RP_TRIG_SRC_CHA_NE",
    rp.RP_TRIG_SRC_CHB_PE:   "RP_TRIG_SRC_CHB_PE",
    rp.RP_TRIG_SRC_CHB_NE:   "RP_TRIG_SRC_CHB_NE",
    rp.RP_TRIG_SRC_EXT_PE:   "RP_TRIG_SRC_EXT_PE",
    rp.RP_TRIG_SRC_EXT_NE:   "RP_TRIG_SRC_EXT_NE",
    rp.RP_TRIG_SRC_AWG_PE:   "RP_TRIG_SRC_AWG_PE",
    rp.RP_TRIG_SRC_AWG_NE:   "RP_TRIG_SRC_AWG_NE",
}

# タイミング定数 (どちらも秒)
_POLL_SLEEP_S   = 1e-4   # トリガ／バッファ充填待ちのループでビジーウェイトしないためのスリープ間隔
_START_SETTLE_S = 1e-3   # rp_AcqStart() 命令が FPGA に到達するのを待つための遅延。
                         # この間に rp_AcqSetTriggerSrc() を呼ぶと取りこぼすことがあるため挟んでいる (本機での経験値)。

# DMA バッファのチャネル別開始アドレス (setup_dma() で書き込まれる)
_DMA_BUF = {}

rp.rp_AcqReset()

# DIO0_N からトリガ信号を出力する設定 (外部機器との同期用)
rp.rp_SetDpinEnableTrigOutput(True)
rp.rp_SetSourceTrigOutput(rp.OUT_TR_ADC)

##### 標準 ACQ 設定 (Option A と確認プロット用) #####
rp.rp_AcqSetDecimation(dec)
rp.rp_AcqSetTriggerLevel(trigger_list[trigger_ch], trig_lvl)
rp.rp_AcqSetTriggerDelay(trig_dly)


def _start_and_wait(trigger_src, fill_checks):
    """1 ショットの取得を開始し、トリガ条件と全バッファの充填を待つ共通処理。"""
    rp.rp_AcqStart()
    time.sleep(_START_SETTLE_S)
    rp.rp_AcqSetTriggerSrc(trigger_src)

    while rp.rp_AcqGetTriggerState()[1] != rp.RP_TRIG_STATE_TRIGGERED:
        time.sleep(_POLL_SLEEP_S)
    for check in fill_checks:
        while not check():
            time.sleep(_POLL_SLEEP_S)


def acqADC(trigger_src):
    """標準 ACQ で 1 ショット取得し、両チャンネルの波形 (numpy 配列, 長さ duty, V) を返す。"""
    _start_and_wait(trigger_src, [lambda: rp.rp_AcqGetBufferFillState()[1]])

    waveform_arr = []
    for channel in (rp.RP_CH_1, rp.RP_CH_2):
        fbuff = rp.fBuffer(N)
        rp.rp_AcqGetOldestDataV(channel, N, fbuff)
        data = np.fromiter(fbuff, dtype=np.float32, count=duty)
        waveform_arr.append(data)

    return waveform_arr


print("標準 ACQ パラメータ設定と acqADC 定義が完了しました。")

## 信号のプロット (Plotly + 標準 ACQ で動作確認)

`acqADC()` でトリガが意図通り効いているか軽く確認します。Plotly はインタラクティブで、ホバーで値が読めたり、ドラッグでズームできたりします。

- **トリガを待たずに即時取得** したい場合: `rp.RP_TRIG_SRC_NOW`
- **トリガを使って取得** する場合: `acq_trig_sour_list[trigger_ch]`

In [ ]:
# 1 ショット取り、2 チャンネル分の波形を Plotly で重ねてプロット
# (paper_style: 16:9, Okabe-Ito 配色, 内向き目盛り, 4 辺枠 が自動適用)
w, h = paper_style.figsize()  # 1280 x 720 = 16:9
waveform_arr = acqADC(acq_trig_sour_list[trigger_ch])
fig = go.Figure()
for i, waveform in enumerate(waveform_arr):
    fig.add_trace(go.Scatter(y=waveform, mode='lines', name=f'ch{i}'))
fig.update_layout(
    xaxis_title='Sample',
    yaxis_title='Voltage [V]',
    width=w, height=h,
)
paper_style.show(fig)


## DMA セットアップと取得関数 (Option B 用)

ここからが本ノートブックの追加部分です。`acqADC_DMA()` は `acqADC()` と同じ「トリガソースを渡して、両チャンネルの numpy 配列リストを受け取る」インタフェースですが、内部では DMA (`rp_AcqAxi*`) を使います。両関数とも、トリガ条件の到来とバッファ充填の待機は共通ヘルパ `_start_and_wait()` が担います。

DMA は標準 ACQ と違って **トリガ前のサンプルを 1 つも残さない設定 (`DMA_TRIG_DELAY = DMA_DATA_SIZE`)** にすると波形の立ち上がりが切れるため、cell 9 で `DMA_PRE_TRIGGER` 分だけ pre-trigger 領域を確保し、`acqADC_DMA()` 内で読み出し位置を巻き戻して読みます。さらに DMA 経路にはハード由来の固定パイプラインレイテンシが乗るので、後段の **較正セル** で実測した `FIXED_OFFSET_SAMPLES` を読み出し位置に追加適用します。

**振る舞いの違い:**

| 項目 | `acqADC()` 標準 | `acqADC_DMA()` DMA |
|---|---|---|
| バッファ長 | 16384 固定、先頭 `duty=256` を抜く | 16384 全サンプルを返す |
| 取得単位 | V (API が変換) | RAW を `_DMA_LSB_VOLTS` で V 換算 |
| データ書き込み先 | カーネルバッファ経由 | **DDR に直接** |
| pre-trigger | バッファ先頭から 30 サンプル目 | `DMA_PRE_TRIGGER` サンプル (cell 9 で設定) |
| 固定レイテンシ補正 | 不要 (API が吸収) | `FIXED_OFFSET_SAMPLES` (較正セルで実測) |

電圧レンジ換算は冒頭 (cell 9) の `INPUT_JUMPER` で `"LV"` (±1V) / `"HV"` (±20V) を切り替えれば、`_DMA_LSB_VOLTS` が `LSB_VOLTS_BY_JUMPER[INPUT_JUMPER]` で派生します。

In [ ]:
# 読み出しバッファをショット間で使い回す (毎回確保しない)
_dma_ibuf = [rp.i16Buffer(DMA_DATA_SIZE), rp.i16Buffer(DMA_DATA_SIZE)]


def _rewind(pos, ch_start, samples_back):
    """ch のリングバッファ内で pos から samples_back サンプルだけ巻き戻した絶対アドレスを返す。"""
    rel = (pos - ch_start - samples_back) % DMA_DATA_SIZE
    return ch_start + rel


def setup_dma():
    """DMA を構成して両チャンネルで有効化する。Option B のループ前に 1 度だけ呼ぶ。"""
    region = rp.rp_AcqAxiGetMemoryRegion()
    axi_start = region[1]            # 予約領域先頭アドレス
    axi_size  = region[2]            # 予約領域サイズ (samples)
    print(f"DMA region: start=0x{axi_start:x} size_samples=0x{axi_size:x}")

    rp.rp_AcqAxiSetDecimationFactor(dec)

    # トリガ後に DMA_TRIG_DELAY サンプルだけ取り続ける
    # (= バッファ全長から DMA_PRE_TRIGGER を引いた長さ。残りは pre-trigger 領域として保持される)
    rp.rp_AcqAxiSetTriggerDelay(rp.RP_CH_1, DMA_TRIG_DELAY)
    rp.rp_AcqAxiSetTriggerDelay(rp.RP_CH_2, DMA_TRIG_DELAY)

    # 予約領域を半分ずつ CH1/CH2 に割り当てる
    ch1_start = axi_start
    ch2_start = axi_start + axi_size // 2
    _DMA_BUF['ch1'] = ch1_start
    _DMA_BUF['ch2'] = ch2_start
    rp.rp_AcqAxiSetBufferSamples(rp.RP_CH_1, ch1_start, DMA_DATA_SIZE)
    rp.rp_AcqAxiSetBufferSamples(rp.RP_CH_2, ch2_start, DMA_DATA_SIZE)

    rp.rp_AcqAxiEnable(rp.RP_CH_1, True)
    rp.rp_AcqAxiEnable(rp.RP_CH_2, True)

    # トリガレベルは標準 ACQ と共有 (rp_AcqSetTriggerLevel)
    rp.rp_AcqSetTriggerLevel(trigger_list[trigger_ch], trig_lvl)
    print("DMA enabled on CH1, CH2")


def teardown_dma():
    """DMA を無効化。Option B のループ終了時に必ず呼ぶ (例外時も finally で)。"""
    rp.rp_AcqAxiEnable(rp.RP_CH_1, False)
    rp.rp_AcqAxiEnable(rp.RP_CH_2, False)


def acqADC_DMA(trigger_src):
    """DMA で 1 ショット取得し、両チャンネルの波形 (numpy float32, 長さ DMA_DATA_SIZE, V) を返す。

    波形は「pre-trigger 領域 (DMA_PRE_TRIGGER サンプル) + post-trigger 領域 (DMA_TRIG_DELAY サンプル)」の
    並びになる。FIXED_OFFSET_SAMPLES が 0 でなければ、その分だけ追加で巻き戻して読み出すことで
    DMA パイプラインの固定レイテンシを補正する。"""
    _start_and_wait(trigger_src, [
        lambda: rp.rp_AcqAxiGetBufferFillState(rp.RP_CH_1)[1],
        lambda: rp.rp_AcqAxiGetBufferFillState(rp.RP_CH_2)[1],
    ])

    rp.rp_AcqStop()

    # トリガ瞬間の書き込みポインタ (= 取得波形の先頭サンプル位置)
    pos1 = rp.rp_AcqAxiGetWritePointerAtTrig(rp.RP_CH_1)[1]
    pos2 = rp.rp_AcqAxiGetWritePointerAtTrig(rp.RP_CH_2)[1]

    # pre-trigger 分 + 固定レイテンシ補正だけ巻き戻して読み出す
    rewind = DMA_PRE_TRIGGER + FIXED_OFFSET_SAMPLES
    read1 = _rewind(pos1, _DMA_BUF['ch1'], rewind)
    read2 = _rewind(pos2, _DMA_BUF['ch2'], rewind)

    rp.rp_AcqAxiGetDataRaw(rp.RP_CH_1, read1, DMA_DATA_SIZE, _dma_ibuf[0].cast())
    rp.rp_AcqAxiGetDataRaw(rp.RP_CH_2, read2, DMA_DATA_SIZE, _dma_ibuf[1].cast())

    ch1 = (np.fromiter(_dma_ibuf[0], dtype=np.int16, count=DMA_DATA_SIZE)
             .astype(np.float32) * _DMA_LSB_VOLTS)
    ch2 = (np.fromiter(_dma_ibuf[1], dtype=np.int16, count=DMA_DATA_SIZE)
             .astype(np.float32) * _DMA_LSB_VOLTS)
    return [ch1, ch2]


print("DMA セットアップと acqADC_DMA 定義が完了しました。")

### DMA 動作確認 (任意、1 ショットだけ取って Plotly で表示)

Option B のループに入る前に、DMA が正しく構成できているか単発で確認します。CH1/CH2 を縦に並べて表示します。

In [ ]:
setup_dma()
try:
    waveform_arr = acqADC_DMA(acq_trig_sour_list[trigger_ch])
    w, h = paper_style.figsize(rows=2)  # 16:9 per panel, 2 行積み = 1280 x 1440
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=('ch0 (DMA)', 'ch1 (DMA)'),
                        vertical_spacing=0.08)
    # pre-trigger + duty サンプルを表示してトリガ前後の様子が両方見えるようにする
    view_n = DMA_PRE_TRIGGER + duty
    for i, waveform in enumerate(waveform_arr):
        fig.add_trace(go.Scatter(y=waveform[:view_n], mode='lines', name=f'ch{i}'),
                      row=i + 1, col=1)
        # トリガ瞬間を縦線で示す (DMA_PRE_TRIGGER の位置)
        fig.add_vline(x=DMA_PRE_TRIGGER, line=dict(color='gray', width=1, dash='dot'),
                      row=i + 1, col=1)
    fig.update_xaxes(title_text='Sample (0 = 波形先頭, トリガ瞬間 = DMA_PRE_TRIGGER)', row=2, col=1)
    fig.update_yaxes(title_text='Voltage [V]')
    fig.update_layout(width=w, height=h, showlegend=False)
    paper_style.show(fig)
    print(f'shape: ch1={waveform_arr[0].shape} ch2={waveform_arr[1].shape}')
    print(f'range ch1: [{waveform_arr[0].min():.4f}, {waveform_arr[0].max():.4f}] V')
    print(f'pre-trigger = {DMA_PRE_TRIGGER}, fixed_offset = {FIXED_OFFSET_SAMPLES} samples')
finally:
    teardown_dma()


## DMA パイプライン固定レイテンシの較正

DMA 経路には ADC→DDR 間のパイプライン遅延（固定値、概ね 5〜20 サンプル程度）が乗るため、`pos_at_trig` から読み始めると「実際にしきい値を切ったサンプル」が `DMA_PRE_TRIGGER` ちょうどではなく、そこから固定オフセット分ズレた位置に現れます。

このセルで較正を行います。手順:

1. **検出器を実信号源として接続** し、本番計測と同じトリガレベル (`trig_lvl`) で `N_CALIB_SHOTS` ショットだけ取得する。
2. 各ショットでしきい値を最初に超えたサンプルの位置から想定位置 `DMA_PRE_TRIGGER` を引き、サンプル単位のオフセットを記録する。
3. 集約値（既定: `median`、ガウス的なジッタに対して mean とほぼ等価で外れ値耐性が高い）を `FIXED_OFFSET_SAMPLES` に書き戻す。

以降の `acqADC_DMA()` は `DMA_PRE_TRIGGER + FIXED_OFFSET_SAMPLES` だけ巻き戻して読み出すので、トリガサンプルが波形内のちょうど `DMA_PRE_TRIGGER` の位置に来ます。

**注意:** 較正値はカーネル内のメモリにのみ残ります。カーネルを再起動した場合や FPGA イメージを更新した場合は再較正してください。較正をスキップしたまま本番計測しても、`FIXED_OFFSET_SAMPLES = 0` のフォールバック値で従来動作と同等に動きます。

In [ ]:
# === DMA パイプラインレイテンシ較正 ===========================================
N_CALIB_SHOTS = 32
CALIB_AGG     = "median"   # "median" / "mean" / "min"

if DMA_PRE_TRIGGER < 32:
    raise RuntimeError(
        f"DMA_PRE_TRIGGER={DMA_PRE_TRIGGER} は較正に小さすぎます。cell 9 で 256 以上に設定してください。"
    )


def _measure_offset(waveform, threshold, expected_idx):
    """波形内で初めて threshold を超えたサンプル位置から expected_idx を引いた値を返す。
    閾値を一度も超えなければ None。"""
    over = np.flatnonzero(waveform > threshold)
    return int(over[0]) - expected_idx if over.size else None


setup_dma()
offsets = []
print(f"較正開始: 最大 {N_CALIB_SHOTS} ショット (Ctrl+C で中断可)")
try:
    try:
        for k in range(N_CALIB_SHOTS):
            wave = acqADC_DMA(acq_trig_sour_list[trigger_ch])[trigger_ch]
            d = _measure_offset(wave, trig_lvl, DMA_PRE_TRIGGER)
            if d is not None:
                offsets.append(d)
            print(f"  shot {k+1:2d}/{N_CALIB_SHOTS}: offset = {d}")
    except KeyboardInterrupt:
        print(f"中断されました ({len(offsets)} ショット使用)")
finally:
    teardown_dma()

if not offsets:
    print("較正失敗: 1 ショットも閾値を超えませんでした。トリガレベルや結線を確認してください。")
    print(f"FIXED_OFFSET_SAMPLES は {FIXED_OFFSET_SAMPLES} のままです。")
else:
    arr = np.asarray(offsets, dtype=np.int64)
    aggregate = {
        "mean":   int(np.round(arr.mean())),
        "median": int(np.median(arr)),
        "min":    int(arr.min()),
    }
    FIXED_OFFSET_SAMPLES = aggregate[CALIB_AGG]
    print()
    print(f"使用ショット数: {len(arr)} / {N_CALIB_SHOTS}")
    print(f"オフセット分布: min={arr.min()}, max={arr.max()}, "
          f"mean={arr.mean():.2f}, median={np.median(arr):.1f}, std={arr.std():.2f} [samples]")
    print(f"採用 ({CALIB_AGG}): FIXED_OFFSET_SAMPLES = {FIXED_OFFSET_SAMPLES} samples "
          f"({FIXED_OFFSET_SAMPLES * 8e-9 * 1e9:.2f} ns @ dec=1)")

## 計測 (Measurement)

ここからが本番のデータ取得です。データは `./data/` ディレクトリに保存されます (無ければ自動作成)。

**ご注意:** 次の 2 つの取得セル (Option A / Option B) は **どちらか一方を選んで実行** してください。

- **Option A** — ピーク値・積分値ロギング (CSV, `.dat`)。**標準 ACQ** を使用。後段集計が軽くて済みます。
- **Option B** — 全波形ロギング (`.npz`)。**DMA** を使用。`DMA_DATA_SIZE = 16384` サンプル/ch を毎ショット保存。

Option A → Option B の順で連続実行する場合、Option B 側で自動的に `setup_dma()` / `teardown_dma()` を呼ぶので追加の切替操作は不要です。

In [ ]:
# === Option A: ピーク値・積分値ロギング (CSV) — 標準 ACQ ===
import json
from pathlib import Path

DATA_DIR = Path('./data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

start_date = datetime.datetime.now()
filename = start_date.strftime('%Y-%m-%d-%H%M%S')
out_path = DATA_DIR / f'{filename}.dat'
meta_path = DATA_DIR / f'{filename}.json'
print(f"計測開始: {out_path}")

meta = {
    'start_datetime':         start_date.isoformat(),
    'mode':                   'option_a_peak_sum_csv',
    'acq_path':               'standard_acq',
    'decimation_macro':       'RP_DEC_1',
    'samples_per_shot':       N,
    'duty':                   duty,
    'trigger_ch_index':       trigger_ch,
    'trigger_src':            TRIG_SRC_NAMES[acq_trig_sour_list[trigger_ch]],
    'trigger_level_V':        trig_lvl,
    'trigger_delay_samples':  trig_dly,
    'input_range_jumper':     INPUT_JUMPER,
    'lsb_to_volts':           _DMA_LSB_VOLTS,
    'csv_columns':            ['totaltime', 'deadtime', 'sum_ch1', 'max_ch1', 'sum_ch2', 'max_ch2'],
}
meta_path.write_text(json.dumps(meta, indent=2))

total_deadtime = 0.0
start_time = time.time()

with out_path.open('w') as f:
    f.write(','.join(meta['csv_columns']) + '\n')
    try:
        while True:
            waveform_arr = acqADC(acq_trig_sour_list[trigger_ch])
            acq_end_time = time.time()
            totaltime = acq_end_time - start_time
            ch1 = waveform_arr[0]
            ch2 = waveform_arr[1]
            data = [
                totaltime,
                total_deadtime,
                float(np.sum(ch1)),
                float(np.max(ch1)),
                float(np.sum(ch2)),
                float(np.max(ch2)),
            ]
            f.write(",".join(map(str, data)) + '\n')
            total_deadtime += time.time() - acq_end_time

    except KeyboardInterrupt:
        print("ユーザによって停止されました。")

### Option B — 全波形ロギング (`.npz`) — **DMA**

このセルは Option A の代替です。各ショットで取得した 2 チャンネル分の波形を **DMA で取得し、メモリ上のリストに溜め込んでから、計測終了時にまとめて 1 つの `.npz` に保存** します。1 ショットあたり `DMA_DATA_SIZE = 16384` サンプル/ch を **そのまま** 残します。

`Ctrl+C` (停止ボタン) でループを抜けると、それまでに溜まったショットを `data/{タイムスタンプ}.npz` に圧縮保存し、取得条件のメモを `data/{タイムスタンプ}.json` に書き出します。これは Option A の `.dat` + `.json` と同じファイル構成です。

**メモリの目安:** 1 ショット ≈ 128 kB (2 ch × 16384 × 4 B)。1000 ショットで ≈ 128 MB なので、Red Pitaya 上では **数千ショットを超える長時間計測** になる場合は Option A (サマリ統計のみ保存) を検討してください。

**Option A と両方を実行する必要はありません。** 依存ライブラリは `numpy` だけで、追加の pip インストールは不要です (HDF5/h5py は使いません)。

In [ ]:
# === Option B: 全波形ロギング (.npz) — DMA ===
import json
from pathlib import Path

DATA_DIR = Path('./data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

start_date = datetime.datetime.now()
filename  = start_date.strftime('%Y-%m-%d-%H%M%S')
out_path  = DATA_DIR / f'{filename}.npz'
meta_path = DATA_DIR / f'{filename}.json'
print(f"計測開始 (DMA): {out_path}")

# ショットごとの波形をメモリに溜める。Ctrl+C 後に numpy 配列にまとめて 1 回だけ保存する。
ch1_list, ch2_list, totaltime_list, deadtime_list = [], [], [], []
total_deadtime = 0.0
start_time = time.time()

setup_dma()
try:
    try:
        while True:
            waveform_arr = acqADC_DMA(acq_trig_sour_list[trigger_ch])
            acq_end_time = time.time()
            totaltime = acq_end_time - start_time

            ch1_list.append(waveform_arr[0])
            ch2_list.append(waveform_arr[1])
            totaltime_list.append(totaltime)
            deadtime_list.append(total_deadtime)

            total_deadtime += time.time() - acq_end_time
    except KeyboardInterrupt:
        print(f"ユーザによって停止されました。{len(ch1_list)} ショットを保存します。")
finally:
    teardown_dma()
    if ch1_list:
        meta = {
            'start_datetime':       start_date.isoformat(),
            'mode':                 'option_b_full_waveform_npz_dma',
            'acq_path':             'dma_axi',
            'decimation_macro':     'RP_DEC_1',
            'samples_per_shot':     DMA_DATA_SIZE,
            'dma_trigger_delay':    DMA_TRIG_DELAY,
            'dma_pre_trigger':      DMA_PRE_TRIGGER,
            'fixed_offset_samples': FIXED_OFFSET_SAMPLES,
            'trigger_ch_index':     trigger_ch,
            'trigger_src':          TRIG_SRC_NAMES[acq_trig_sour_list[trigger_ch]],
            'trigger_level_V':      trig_lvl,
            'input_range_jumper':   INPUT_JUMPER,
            'lsb_to_volts':         _DMA_LSB_VOLTS,
            'sample_dt_s':          8e-9,           # dec=1 (125 MSPS) の 1 サンプル周期 [秒]
            'n_shots':              len(ch1_list),
        }
        np.savez_compressed(
            out_path,
            ch1=np.asarray(ch1_list,        dtype=np.float32),
            ch2=np.asarray(ch2_list,        dtype=np.float32),
            totaltime=np.asarray(totaltime_list, dtype=np.float64),
            deadtime=np.asarray(deadtime_list,   dtype=np.float64),
        )
        meta_path.write_text(json.dumps(meta, indent=2))
        print(f"保存完了: {out_path}")
        print(f"           {meta_path}")
    print("DMA を無効化しました。")

## クリーンアップ

ノートブック全体を終了する前に、念のため DMA を無効化し、リソースを解放しておきます。

(Option B が正常終了/`KeyboardInterrupt` した場合は既に `teardown_dma()` が走っていますが、二重に呼んでも害はありません。)

In [ ]:
try:
    teardown_dma()
except Exception as e:
    print(f"teardown_dma() スキップ: {e}")
rp.rp_Release()
print("終了しました。")

## トラブルシューティング

### `fig.show()` を実行しても何も出ない
JupyterLab の場合、Plotly 5.x は追加 extension なしで描画できます。それでも空白の場合は以下を試してください。

```python
import plotly.io as pio
pio.renderers.default = "notebook_connected"   # クラシック Notebook 向け
# または
pio.renderers.default = "jupyterlab"           # JupyterLab 向け
```

### それでも描画されない / オフラインで使いたい
HTML として保存してブラウザで開く方法が確実です。

```python
fig.write_html('shot.html')   # ./shot.html を生成。ブラウザで開く
```

### `pip install` 中に numpy がビルドされ始める
Plotly のサブ依存 (`narwhals` など) が新しい numpy を要求して再ビルドが走ることがあります。冒頭で書いたように **必ず `--no-deps` を付けて plotly 本体を入れ、足りない純 Python 依存だけ別途入れてください**。

### DMA で `Already enabled` エラーが出る
前回の実行で `teardown_dma()` が呼ばれずに残っているケースです。クリーンアップセル (一番下) を一度実行してから setup し直してください。